# TCAA Experiment — Google Colab（两实验 · 图/文分离留存）

**TCAA** = Token-Consumption Amplification Attack：联邦微调（FFT）里一个恶意客户端上传精心构造的 LoRA 更新，使聚合后的因果 LLM 在**带触发词的输入**上生成更多 token（更耗算力/成本），同时保持输出正确、并尽量躲过服务器检测。

本 notebook 分成两个**互相区分**的实验，每个实验的**图表**与**可复制文本**各自独立成格，方便分析与复制：

- 🅱　**实验 B · 多轮联邦 FL** — 放大是否随轮次累积/存活（durability）、多轮下效用是否被保持（对**原始骨干**的 ppl/ROUGE）、逐轮隐蔽。**主结果**。
- 🅲　**实验 C · Pareto 扫描** — 放大 vs 隐蔽预算的权衡前沿（每个网格点本身就是一次单轮 de-risk）。

> 说明：单轮 de-risk 曾是独立的「实验 A」，但它只是 C 网格里的一个点、也是 B 的 round 0，冗余，已并入 B/C，不再单列。

每个实验固定四步：**配置 → 运行 → 📊 图表导出 → 📋 文本导出**。**图表格只画图、文本格只出可复制文本**；跑完把各实验「📋 文本导出」格的输出整块复制回传即可分析。先跑「准备」段（Step 0–3）与下方**共用基础配置**，两个实验都复用它。

## Step 0: Fetch Code（获取代码 · 三选一）

含 `tcaa/` 的代码目前只在你本地，需先让 Colab 能访问。**任选一种**：

- **A｜GitHub（推荐）**：把本仓库 push 到你自己的 GitHub，然后把下面 `REPO_URL` 改成你的地址。
- **B｜Google Drive**：把整个 `TCA-Attacker` 文件夹放到 Drive，运行下面「可选：挂载 Drive」单元并改成你的路径。
- **C｜手动上传**：把整个文件夹拖到 Colab 左侧文件区。

下面的取码单元会**自动按 A/B/C 顺序探测**，找到 `tcaa/` 即用。

In [ ]:
# （可选 · 方式 B）把代码放在 Google Drive 时才用：取消下面两行注释并改成你的实际路径，再运行本单元。
# from google.colab import drive; drive.mount('/content/drive')
# %cd '/content/drive/MyDrive/TCA-Attacker'
print('如走 Google Drive：请取消上面两行注释并改路径；否则跳过本单元。')

In [ ]:
# 自动获取代码：C(当前目录已有) -> A(GitHub 克隆)。找到 tcaa/ 即停。
import os, sys, subprocess
from pathlib import Path

# 👉 方式 A：改成你自己的 fork（必须包含 tcaa/ 包）。默认按 git user 猜测，请自行确认。
REPO_URL = 'https://github.com/GuangLun2000/TCA-Attacker.git'

def find_repo_root():
    for base in [Path('.'), *sorted(Path('.').glob('*')), *sorted(Path('.').glob('*/*'))]:
        if base.is_dir() and (base / 'tcaa' / 'phase0_runner.py').exists():
            return base
    return None

root = find_repo_root()
if root is None and '<' not in REPO_URL:
    print(f'📥 未在当前环境找到 tcaa/，尝试克隆 {REPO_URL} ...')
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
        root = find_repo_root()
    except subprocess.CalledProcessError:
        print('❌ 克隆失败（仓库私有/地址错/无权限）。')

if root is None:
    raise SystemExit(
        '❌ 未找到 tcaa/ 包。请用以下任一方式让 Colab 拿到代码：\n'
        '  A) 把仓库 push 到你的 GitHub，并把 REPO_URL 改对；\n'
        '  B) 把文件夹放 Google Drive，运行上一单元挂载并 cd 过去；\n'
        '  C) 把整个文件夹上传到 Colab 左侧文件区，然后重跑本单元。')

os.chdir(root.resolve())
sys.path.insert(0, str(Path('.').resolve()))
print('✅ 使用仓库:', Path('.').resolve())
assert Path('tcaa/phase0_runner.py').exists()

## Step 1: Install Dependencies（安装依赖）

In [ ]:
# Colab 已自带 torch；补装 peft + datasets + 较新的 transformers。
%pip install -q "transformers>=4.37,<5" "peft>=0.6" "datasets>=2" tqdm
# Colab 预装的旧版 torchao(0.10) 会让新版 peft 的 LoRA 分发器报 ImportError；本项目不用 torchao，卸载即可。
!pip uninstall -y torchao 2>/dev/null || true
print('✅ 依赖安装完成（已移除不兼容的 torchao）。')

## Step 2: Verify Files and GPU（检查文件与 GPU）

In [ ]:
import torch
from pathlib import Path

need = ['tcaa/phase0_runner.py', 'tcaa/length_surrogate.py', 'tcaa/cost_model.py',
        'tcaa/causal_model.py', 'tcaa/gen_data.py', 'tcaa/stealth.py', 'tcaa/metrics.py',
        'tcaa/alm.py', 'tcaa/visualize.py', 'tcaa/fl_runner.py', 'tcaa/pareto_runner.py']
missing = [f for f in need if not Path(f).exists()]
print('⚠️ 缺少文件:', missing) if missing else print('✅ tcaa/ 关键文件齐全。')

print('\nPyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} ({p.total_memory/1e9:.1f} GB)')
else:
    print('⚠️ 未检测到 GPU。Runtime → Change runtime type → GPU（不然会很慢）')

## Step 3: Sanity Check（单元测试，快速自检环境）

In [ ]:
!python -m tcaa.tests.test_length_surrogate
!python -m tcaa.tests.test_alm
!python -m tcaa.tests.test_stealth_matches_server

**共用基础配置**　默认 **Qwen2.5-0.5B（base）+ Alpaca**，正式规模。Alpaca 输出长度自由，最适合展示 token 膨胀。换骨干/数据集见注释；实验 B / C 都复用这份配置。

In [ ]:
TCAA_CONFIG = {
    "experiment_name": "tcaa_qwen25_alpaca",
    "backbone": "Qwen/Qwen2.5-0.5B",         # base 版做联邦微调；仅 decoder-only（会校验）
    "source": "alpaca",                       # 'alpaca'(指令,默认) | 'dolly' | 'xsum' | 'cnn_dailymail'
    "reference_source": "dataset",            # 或 'benign_verbose'（Spec §3 source ii）

    # —— 联邦设置（与 AugMP 约定一致，便于对比）——
    "num_clients": 5, "num_attackers": 1, "local_epochs": 2,
    "dirichlet_alpha": 0.3, "server_lr": 1.0,
    "client_lr": 1e-4, "batch_size": 8, "grad_clip_norm": 1.0,
    "warmup_steps": 0,                        # 预训练骨干无需 warm-up

    # —— TCAA 攻击（含逻辑改进）——
    "gamma": 1.0, "attacker_lr": 1e-4, "attacker_steps": 300,
    "gamma_clean": 0.5,                       # clean 长度锚：抑制泄漏、提升触发选择性（单边 hinge 到基线）
    "use_onpolicy_length": True,              # 沿模型自身 greedy 续写优化 E[L]（对齐"目标=推理测量"，修 exposure bias）
    "onpolicy_horizon": 256,                  # on-policy 续写步数 = 消耗杠杆(越大越长/越慢)；应 << max_new_tokens
    "use_fallback_surrogate": False,
    # —— 反重复(让"变长"=有用地变长,而非复读循环;直接改善效用+隐蔽)——
    "gamma_rep": 0.2,                         # 可微反重复惩罚权重；若 rep_tau 仍高→调大(0.4/0.6)
    "no_repeat_ngram_size": 3,                # on-policy 续写里禁止重复 n-gram(轨迹更多样)
    "stealth_use_pairwise_cosine": True,      # 隐蔽判决用与 ALM 一致的成对余弦(口径对齐)

    # —— 成本模型 / 生成 ——
    # c_f/c_a 只有比值影响超线性阈值；如需硬件标定用 cost_model.calibrate_coefficients(d_model)
    "c_f": 1.0, "c_a": 1.0, "max_new_tokens": 1024,  # 抬到 1024：露出真实(被截断)长度；残余截断由"去删失估计"补正
    "num_dump_examples": 4,                   # 定性样例数/split（触发→长文、clean→正常）

    # —— 隐蔽阈值：None = 用良性包络（d_T=良性最大距离，δ_T=良性最小余弦）——
    "d_T": None, "delta_T": None,

    # —— 数据规模（正式）——
    "pool_size": 512, "eval_size": 64,
    "lora_r": 8, "lora_alpha": 16,
}

# ===== 常用替换（取消注释即可）=====
# 更大骨干（需 L4/A100）：
# TCAA_CONFIG.update({"backbone": "Qwen/Qwen2.5-1.5B"})
# TCAA_CONFIG.update({"backbone": "Qwen/Qwen2.5-3B"})
# 单轮一般无需效用地板（clean 本就中性）；如要开启：TCAA_CONFIG.update({"kd_clean_weight": 1.0})
# 换数据集：
# TCAA_CONFIG.update({"source": "dolly"})       # 另一指令数据集
# TCAA_CONFIG.update({"source": "xsum"})        # 摘要任务
# 详细但正确的参考（source ii，进一步缓解短参考问题）：
# TCAA_CONFIG.update({"reference_source": "benign_verbose"})
# on-policy 太慢时改回 teacher-forced（更快、但目标与推理略脱节）：
# TCAA_CONFIG.update({"use_onpolicy_length": False})
# 更快试跑：TCAA_CONFIG.update({"pool_size":128,"eval_size":32,"attacker_steps":150,"local_epochs":1,"onpolicy_horizon":64})
# 更强攻击/更大消耗：提高 onpolicy_horizon(如 384)、gamma、attacker_steps；若 rep 高→提高 gamma_rep；盯 TUNING HINTS

print('配置就绪:', TCAA_CONFIG['backbone'], '+', TCAA_CONFIG['source'],
      '| pool', TCAA_CONFIG['pool_size'], '| steps', TCAA_CONFIG['attacker_steps'],
      '| on-policy', TCAA_CONFIG['use_onpolicy_length'], '| max_new', TCAA_CONFIG['max_new_tokens'])

<div style="border-left:7px solid #009E73;background:#e9f7f2;padding:12px 16px;border-radius:8px;">
<span style="font-size:20px;font-weight:800;color:#009E73;">🅱　实验 B · 多轮联邦 FL（durability + 效用保持）</span><br>
<span style="color:#333;font-size:13px;">放大是否随通信轮次累积/存活、每轮是否隐蔽、多轮下 clean 困惑度是否被保持（ppl_ratio≈1）。较重，默认已调小；复用上方共用基础配置。</span>
</div>

**B.1 配置 + 运行**　5 benign + 2 attackers、多轮 FedAvg 采样；并行跑一条良性-only 全局作成本/困惑度基线。运行中 durability/stealth 表会打印（留存）。

In [ ]:
import time
from tcaa.fl_runner import run_fl, default_fl_config

FL_CONFIG = default_fl_config()
FL_CONFIG.update({
    "backbone": TCAA_CONFIG["backbone"], "source": TCAA_CONFIG["source"],
    "num_clients": 7, "num_attackers": 2,   # 5 benign + 2 coordinated attackers
    # —— 效用地板（核心修复）：多轮下把 clean 分布锚到冻结骨干，防 ppl 累积漂移 ——
    #    ppl_ratio 仍随轮次上升 → 调大(2/4)；放大被压垮 → 调小(0.5)；0 = 关闭。
    "kd_clean_weight": 1.0,
    # —— 默认调小以便一次跑完（可上调）——
    "num_rounds": 20, "clients_per_round": 5, "measure_every": 5,
    "pool_size": 1500, "eval_size": 128, "attacker_steps": 60,
    # cap/horizon/gamma_rep 已从 default_fl_config 继承(1024/256/0.2)；如太慢可下调:
    # "max_new_tokens": 512, "onpolicy_horizon": 160,
})
# 正式规模（论文级，较慢）：
# FL_CONFIG.update({"num_rounds": 50, "pool_size": 4000, "eval_size": 256, "attacker_steps": 100})

print('🚀 实验 B · 多轮 FL 开始 ...'); print('=' * 60)
print(f"   kd_clean_weight={FL_CONFIG['kd_clean_weight']} (效用地板)  "
      f"gamma={FL_CONFIG['gamma']}  gamma_rep={FL_CONFIG['gamma_rep']}  horizon={FL_CONFIG['onpolicy_horizon']}  "
      f"rounds={FL_CONFIG['num_rounds']}  cap={FL_CONFIG['max_new_tokens']}")
t0 = time.time()
fl_results = run_fl(FL_CONFIG)      # durability / utility(ppl_ratio) / stealth 表随运行打印（留存）
print(f'\n✅ 实验 B 多轮完成，用时 {(time.time()-t0)/60:.1f} 分钟。')


#### 📊 B.2 图表导出　·　仅图
3 张图：durability（放大累积）+ **效用保持/退化**（ppl_ratio 带 ±5% 保持带、trunc、rep）+ 逐轮隐蔽。

In [ ]:
%matplotlib inline
from tcaa.visualize import render_fl_report
# 仅图 · 3 张：durability(放大累积) + 效用保持/退化(ppl_ratio, trunc, rep) + 逐轮隐蔽
_ = render_fl_report(fl_results)


#### 📋 B.3 文本导出　·　仅文本（全选复制本格输出回传）
多轮逐轮表：amp / med / sel / trunc / rep / **ppl_ratio** / stealth，+ durability 与效用趋势。

In [ ]:
# 实验 B 文本导出（纯文本，全选复制本格输出回传）：多轮 durability + 效用(ppl_ratio) + 退化。
from tcaa.visualize import feedback_digest
_ = feedback_digest(fl=fl_results)


<div style="border-left:7px solid #CC79A7;background:#f9edf4;padding:12px 16px;border-radius:8px;">
<span style="font-size:20px;font-weight:800;color:#CC79A7;">🅲　实验 C · Pareto 扫描（放大 vs 隐蔽预算）</span><br>
<span style="color:#333;font-size:13px;">扫 γ × κ，画放大-隐蔽前沿 + 可用前沿（放大 vs 效用）。较重，默认小网格；复用上方共用基础配置。</span>
</div>

**C.1 配置 + 运行**　扫 γ×κ 网格，每点一次约束版 Phase-0；前沿表随运行打印（留存）。

In [ ]:
import time
from tcaa.pareto_runner import run_pareto

PARETO_BASE = dict(TCAA_CONFIG)
PARETO_BASE.update({          # 单点调小，保持网格可跑完
    "pool_size": 256, "eval_size": 48, "attacker_steps": 120,
    "results_subdir": "tcaa_phase0",
})
print('🚀 实验 C · Pareto 扫描开始 ...'); print('=' * 60)
t0 = time.time()
pareto = run_pareto(PARETO_BASE, gammas=[1.0, 2.0], gamma_cleans=[0.5],
                    kappas=[0.6, 0.8, 1.0])   # 前沿表随运行打印（留存）
print(f'\n✅ 实验 C Pareto 完成，用时 {(time.time()-t0)/60:.1f} 分钟。')


#### 📊 C.2 图表导出　·　仅图
3 张图：放大-隐蔽前沿 + κ 权衡 + **可用前沿**（放大 vs 效用保持，点大小∝选择性）。

In [ ]:
%matplotlib inline
from tcaa.visualize import render_pareto_report
# 仅图 · 3 张：放大-隐蔽前沿 + κ 权衡 + 可用前沿(放大 vs 效用保持)
_ = render_pareto_report(pareto)


#### 📋 C.3 文本导出　·　仅文本（全选复制本格输出回传）
每个网格点：gamma / kappa / amp / med / clean / sel / **ppl_r** / dist·d_T / JOINT。

In [ ]:
# 实验 C 文本导出（纯文本，全选复制本格输出回传）：Pareto 每点 amp/clean/sel/ppl_r/dist/d_T/JOINT。
from tcaa.visualize import feedback_digest
_ = feedback_digest(pareto=pareto)


## 📋 一站式诊断导出（复制本格**全部输出**回传即可分析/迭代）
把两个实验的**全部过程数据 + 结果数据**汇成**一个**可复制文本块：**配置** · **B 每轮完整指标表**(放大/去删失/有效/vs 原始骨干/效用/ROUGE/隐蔽) · **逐轮隐蔽轨迹** · **一段攻击者单轮内优化轨迹**(L_mal·E[L]·q_eos·rep·ALM) · **最终解码样例**(看输出是否又长又连贯,判断反重复是否奏效) · **自动调参建议**。跑完把本格输出整块复制回传即可；只跑了部分实验也行，缺失项自动跳过。

In [ ]:
# 一站式诊断导出：复制【本格全部输出】回传给我 —— 含配置 / B 每轮表(放大·去删失·有效·vs原始骨干·效用·ROUGE·隐蔽)
# / 逐轮隐蔽轨迹 / 一段攻击者单轮内优化轨迹(L_mal·E[L]·q_eos·rep·ALM) / 最终解码样例(判断"变长是否有用/连贯") / 自动调参建议。
# 只跑了部分实验也行，缺失项自动跳过。
from tcaa.visualize import full_report
_ = full_report(
    fl=globals().get('fl_results'),       # 实验 B（多轮 FL）
    pareto=globals().get('pareto'),       # 实验 C（Pareto）
    phase0=globals().get('results'),      # 兼容：若手动跑过单轮 A 则纳入，否则自动跳过
)

## Step 9: Download Results（打包下载 · 离线备份）

页面留存已够用；这里再把 `results/`（phase0 / 多轮 FL / pareto 三段）打包成 zip 供离线备份。

In [ ]:
import zipfile
from pathlib import Path

rd = Path('results')
zp = 'tcaa_results.zip'
if rd.exists():
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in rd.rglob('*'):
            if f.is_file():
                z.write(f, f.relative_to(rd))
    print('✅ 已打包 results/（phase0 + fl + pareto）:', zp)
    try:
        from google.colab import files
        files.download(zp)
    except Exception:
        print('本地打开:', Path(zp).resolve())
else:
    print('⚠️ 未找到 results/，请先运行前面的实验单元。')

## Step 10: 自动释放 GPU（运行结束 30 秒后）

**释放前请先 Ctrl/⌘+S 保存 notebook**，否则内联的表与图不会写回 `.ipynb`。本单元**倒计时 30 秒**再自动释放 Colab GPU（`runtime.unassign()`），避免空占浪费；释放会**断开当前 runtime**，务必让 Step 9 下载先完成。

> 想继续使用 GPU：在这 30 秒倒计时内点击 Colab 工具栏的 **■（停止本单元）** 即可取消释放。

In [ ]:
# 运行结束 → 倒计时 60 秒 → 自动释放 Colab GPU（避免空占浪费）。
# ⚠️ 释放前请先 Ctrl/⌘+S 保存 notebook，确保内联的表与图写回 .ipynb。
# 取消自动释放：在这 60 秒内点击工具栏的 ■ 停止本单元即可。
import time

WAIT_SECONDS = 60
print(f'✅ 实验与下载已完成。请先 Ctrl/⌘+S 保存；将在 {WAIT_SECONDS} 秒后自动释放 GPU。')
print('   如需继续使用 GPU：现在点击工具栏的 ■（停止本单元）即可取消释放。')
try:
    for i in range(WAIT_SECONDS, 0, -1):
        print(f'\r   {i:2d} 秒后释放 GPU ...  ', end='', flush=True)
        time.sleep(1)
    print('\n🔌 正在释放 Colab GPU runtime ...')
    from google.colab import runtime
    runtime.unassign()          # 断开并释放 GPU（会结束当前会话）
except KeyboardInterrupt:
    print('\n⏹️  已取消自动释放，GPU 继续保留。')
except Exception as e:
    print('\n（非 Colab 环境或释放失败，可忽略）:', e)